# Filter-Based Feature Selection Runtime Analysis

This notebook is the main experiment entrypoint. It compares mRMR, ReliefF, and CFS-Greedy on stratified Gisette subsamples from 10% to 100% of the labeled dataset.

## 1. Imports and Constants

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from algorithms import select_cfs_greedy, select_mrmr, select_relieff
from utils.data_loader import load_dataset

DATASET_NAME = "gisette"
RANDOM_STATE = 42
FRACTIONS = [round(value / 10, 1) for value in range(1, 11)]
TOP_K_RATIO = 0.10
TEST_SIZE = 0.20

RESULTS_DIR = Path("results") / "filter_analysis"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## 2. Shared Helpers

In [ ]:
def matrix_nonzero_count(x) -> int:
    if issparse(x):
        return int(x.nnz)
    return int(np.count_nonzero(x))


def matrix_density(x) -> float:
    total_cells = x.shape[0] * x.shape[1]
    if total_cells == 0:
        return 0.0
    return matrix_nonzero_count(x) / total_cells


def class_distribution_text(y: np.ndarray) -> str:
    labels, counts = np.unique(y, return_counts=True)
    distribution = {str(label): int(count) for label, count in zip(labels, counts)}
    return json.dumps(distribution, sort_keys=True)


def compact_metadata_text(metadata: dict[str, object]) -> str:
    compact: dict[str, object] = {}
    for key, value in metadata.items():
        if isinstance(value, np.ndarray):
            compact[f"{key}_shape"] = list(value.shape)
            continue
        if isinstance(value, list) and len(value) > 20:
            compact[f"{key}_length"] = len(value)
            continue
        compact[key] = value
    return json.dumps(compact, default=str, sort_keys=True)


def selected_indices_text(indices: np.ndarray) -> str:
    return ";".join(str(int(index)) for index in indices)


def make_nested_stratified_subsets(
    y: np.ndarray,
    fractions: list[float] = FRACTIONS,
    random_state: int = RANDOM_STATE,
) -> dict[float, np.ndarray]:
    rng = np.random.default_rng(random_state)
    shuffled_by_class: dict[object, np.ndarray] = {}
    for label in np.unique(y):
        class_indices = np.flatnonzero(y == label)
        shuffled_by_class[label] = rng.permutation(class_indices)

    subsets: dict[float, np.ndarray] = {}
    for fraction in fractions:
        selected_parts: list[np.ndarray] = []
        for class_indices in shuffled_by_class.values():
            selected_count = int(round(len(class_indices) * fraction))
            selected_count = max(1, min(selected_count, len(class_indices)))
            selected_parts.append(class_indices[:selected_count])
        selected_indices = np.concatenate(selected_parts)
        selected_indices.sort()
        subsets[fraction] = selected_indices
    return subsets


def train_test_scale(x_subset, y_subset, random_state: int):
    x_train, x_test, y_train, y_test = train_test_split(
        x_subset,
        y_subset,
        test_size=TEST_SIZE,
        random_state=random_state,
        stratify=y_subset,
    )
    scaler = StandardScaler(with_mean=not issparse(x_train))
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    return x_train_scaled, x_test_scaled, y_train, y_test


def train_and_evaluate_linear_svm(x_train, y_train, x_test, y_test, mask: np.ndarray | None = None) -> tuple[float, float]:
    if mask is not None:
        mask = mask.astype(bool)
        x_train = x_train[:, mask]
        x_test = x_test[:, mask]

    model = SGDClassifier(
        loss="hinge",
        alpha=0.0001,
        max_iter=1000,
        tol=1e-3,
        random_state=RANDOM_STATE,
    )
    start = time.perf_counter()
    model.fit(x_train, y_train)
    train_time_seconds = time.perf_counter() - start
    predictions = model.predict(x_test)
    test_accuracy = accuracy_score(y_test, predictions)
    return train_time_seconds, test_accuracy


## 3. Load Gisette and Summarize Input Sizes

In [ ]:
x, y, feature_names = load_dataset(DATASET_NAME)
if issparse(x):
    x = x.tocsr()

n_rows, n_features = x.shape
TOP_K = max(1, int(round(n_features * TOP_K_RATIO)))
subsets = make_nested_stratified_subsets(y)

summary_rows = []
for fraction, indices in subsets.items():
    x_subset = x[indices]
    y_subset = y[indices]
    summary_rows.append(
        {
            "dataset": DATASET_NAME,
            "fraction": fraction,
            "rows": int(x_subset.shape[0]),
            "features": int(x_subset.shape[1]),
            "input_size": int(x_subset.shape[0] * x_subset.shape[1]),
            "nonzero_count": matrix_nonzero_count(x_subset),
            "density": matrix_density(x_subset),
            "class_distribution": class_distribution_text(y_subset),
        }
    )

subsample_summary = pd.DataFrame(summary_rows)
subsample_summary.to_csv(RESULTS_DIR / "gisette_subsample_summary.csv", index=False)
print(f"Top-k feature cap: {TOP_K} of {n_features} features")
display(subsample_summary)


## 4. Full-Feature Baseline Linear SVM

In [ ]:
x_train_full, x_test_full, y_train_full, y_test_full = train_test_scale(x, y, RANDOM_STATE)
baseline_train_time, baseline_accuracy = train_and_evaluate_linear_svm(
    x_train_full,
    y_train_full,
    x_test_full,
    y_test_full,
    mask=None,
)

baseline_result = pd.DataFrame(
    [
        {
            "model": "Baseline",
            "fraction": 1.0,
            "rows": int(x.shape[0]),
            "features": int(x.shape[1]),
            "input_size": int(x.shape[0] * x.shape[1]),
            "selected_count": int(x.shape[1]),
            "svm_train_time_seconds": baseline_train_time,
            "test_accuracy": baseline_accuracy,
        }
    ]
)
baseline_result.to_csv(RESULTS_DIR / "baseline_result.csv", index=False)
display(baseline_result)


## 5. Run Filter Feature Selection

In [ ]:
ALGORITHMS = {
    "mRMR": lambda x_train, y_train: select_mrmr(x_train, y_train, top_k=TOP_K),
    "ReliefF": lambda x_train, y_train: select_relieff(x_train, y_train, top_k=TOP_K, random_state=RANDOM_STATE),
    "CFS-Greedy": lambda x_train, y_train: select_cfs_greedy(x_train, y_train, max_features=TOP_K),
}

result_rows: list[dict[str, object]] = []
for fraction_index, (fraction, indices) in enumerate(subsets.items(), start=1):
    x_subset = x[indices]
    y_subset = y[indices]
    x_train, x_test, y_train, y_test = train_test_scale(
        x_subset,
        y_subset,
        RANDOM_STATE + fraction_index,
    )

    for algorithm_name, selector in ALGORITHMS.items():
        print(f"Running {algorithm_name} on fraction={fraction:.1f}, rows={x_subset.shape[0]}", flush=True)
        selection_start = time.perf_counter()
        selection_result = selector(x_train, y_train)
        selection_runtime = time.perf_counter() - selection_start

        svm_train_time, test_accuracy = train_and_evaluate_linear_svm(
            x_train,
            y_train,
            x_test,
            y_test,
            selection_result.mask,
        )
        total_runtime = selection_runtime + svm_train_time
        selected_count = int(selection_result.mask.sum())

        result_rows.append(
            {
                "dataset": DATASET_NAME,
                "algorithm": algorithm_name,
                "fraction": fraction,
                "rows": int(x_subset.shape[0]),
                "features": int(x_subset.shape[1]),
                "input_size": int(x_subset.shape[0] * x_subset.shape[1]),
                "nonzero_count": matrix_nonzero_count(x_subset),
                "density": matrix_density(x_subset),
                "selected_count": selected_count,
                "selection_runtime_seconds": selection_runtime,
                "svm_train_time_seconds": svm_train_time,
                "total_runtime_seconds": total_runtime,
                "test_accuracy": test_accuracy,
                "selected_feature_indices": selected_indices_text(selection_result.selected_indices),
                "metadata": compact_metadata_text(selection_result.metadata),
            }
        )
        pd.DataFrame(result_rows).to_csv(RESULTS_DIR / "filter_feature_selection_results.csv", index=False)
        print(
            f"  selected={selected_count}, selection_time={selection_runtime:.4f}s, "
            f"svm_time={svm_train_time:.4f}s, accuracy={test_accuracy:.4f}",
            flush=True,
        )

filter_results = pd.DataFrame(result_rows)
display(filter_results)


## 6. Plot Execution Time, Selected Features, and Best Accuracy

In [ ]:
filter_results = pd.read_csv(RESULTS_DIR / "filter_feature_selection_results.csv")
baseline_result = pd.read_csv(RESULTS_DIR / "baseline_result.csv")

figure, axis = plt.subplots(figsize=(10, 6))
for algorithm_name, algorithm_results in filter_results.groupby("algorithm"):
    algorithm_results = algorithm_results.sort_values("input_size")
    axis.plot(
        algorithm_results["input_size"],
        algorithm_results["selection_runtime_seconds"],
        marker="o",
        label=algorithm_name,
    )
axis.set_title("Feature Selection Execution Time vs Input Size")
axis.set_xlabel("Input size (rows x features)")
axis.set_ylabel("Selection execution time (seconds)")
axis.grid(True, alpha=0.3)
axis.legend()
figure.tight_layout()
execution_plot_path = RESULTS_DIR / "execution_time_vs_input_size.png"
figure.savefig(execution_plot_path, dpi=200)
plt.show()
print(f"Saved plot: {execution_plot_path}")

figure, axis = plt.subplots(figsize=(10, 6))
for algorithm_name, algorithm_results in filter_results.groupby("algorithm"):
    algorithm_results = algorithm_results.sort_values("input_size")
    axis.plot(
        algorithm_results["input_size"],
        algorithm_results["selected_count"],
        marker="o",
        label=algorithm_name,
    )
axis.set_title("Selected Feature Count vs Input Size")
axis.set_xlabel("Input size (rows x features)")
axis.set_ylabel("Selected feature count")
axis.grid(True, alpha=0.3)
axis.legend()
figure.tight_layout()
selected_plot_path = RESULTS_DIR / "selected_features_vs_input_size.png"
figure.savefig(selected_plot_path, dpi=200)
plt.show()
print(f"Saved plot: {selected_plot_path}")

best_rows = [
    {
        "model": "Baseline",
        "test_accuracy": float(baseline_result.loc[0, "test_accuracy"]),
        "input_size": int(baseline_result.loc[0, "input_size"]),
    }
]
for algorithm_name, algorithm_results in filter_results.groupby("algorithm"):
    best_row = algorithm_results.sort_values("test_accuracy", ascending=False).iloc[0]
    best_rows.append(
        {
            "model": algorithm_name,
            "test_accuracy": float(best_row["test_accuracy"]),
            "input_size": int(best_row["input_size"]),
        }
    )

best_accuracy = pd.DataFrame(best_rows)
best_accuracy.to_csv(RESULTS_DIR / "best_accuracy_summary.csv", index=False)

figure, axis = plt.subplots(figsize=(10, 6))
bars = axis.bar(best_accuracy["model"], best_accuracy["test_accuracy"])
axis.set_title("Best Test Accuracy by Model")
axis.set_xlabel("Model")
axis.set_ylabel("Test accuracy")
axis.set_ylim(0, min(1.0, max(best_accuracy["test_accuracy"]) + 0.05))
axis.grid(True, axis="y", alpha=0.3)
for bar, (_, row) in zip(bars, best_accuracy.iterrows()):
    axis.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{row['test_accuracy']:.4f}\ninput={int(row['input_size'])}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
figure.tight_layout()
accuracy_plot_path = RESULTS_DIR / "best_accuracy_comparison.png"
figure.savefig(accuracy_plot_path, dpi=200)
plt.show()
print(f"Saved plot: {accuracy_plot_path}")
display(best_accuracy)
